# [시장분석] 잠실 리센츠 가격이면 살 수 있는 강남·서초 아파트

강남·서초 전체 단지의 면적 타입을 순회해 실제 전용 83~85㎡를 찾고, 2026년 7월 KB 일반가가 리센츠 동일 면적대 중위값 33.5억 원 대비 ±5%인 단지를 HTML 표와 CSV로 만듭니다. 모든 단지의 복수 타입은 중위값으로 통합합니다.


In [ ]:
# @title 비교표를 생성하세요
"""리센츠와 가격이 비슷한 강남·서초 전용 83~85㎡ 아파트를 비교한다.

KB부동산 단지별 매매 일반가와 기본정보를 사용한다. 리센츠의 전용
83~85㎡ 타입 일반가 중위값을 기준으로 ±5% 범위에 드는 단지를 선별한다.
2026년 7월 월간 시세를 사용하고 복수 타입은 단지별 중위값으로 통합한다.
"""

from __future__ import annotations

import os
import statistics
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import FileLink, HTML, display


REFRESH_KB_DATA = False  # @param {type:"boolean"}

API_COMPLEX = "https://api.kbland.kr/land-complex"
API_PRICE = "https://api.kbland.kr/land-price"
TARGET_YEAR_MONTH = "202607"
TARGET_PRICE_DATE = "2026-07-31"
AREA_MIN = 83.0
AREA_MAX = 85.0
PRICE_TOLERANCE_RATE = 0.05
REFERENCE_COMPLEX_ID = 15524
REFERENCE_PRICE_MANWON = 335_000
REFERENCE_PRICE_EOK = REFERENCE_PRICE_MANWON / 10_000
LOWER_PRICE_EOK = REFERENCE_PRICE_EOK * (1 - PRICE_TOLERANCE_RATE)
UPPER_PRICE_EOK = REFERENCE_PRICE_EOK * (1 + PRICE_TOLERANCE_RATE)

IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
COMPLEX_CACHE = OUTPUT_DIR / "kb_gangnam_seocho_complexes.csv"
TYPE_PRICE_CACHE = OUTPUT_DIR / "kb_gangnam_seocho_83_85_types_202607.csv"
RESULT_PATH = OUTPUT_DIR / "recenz_vs_gangnam_kb_price.csv"

LEGAL_DONGS = {
    "강남구": {
        "역삼동": "1168010100", "개포동": "1168010300",
        "청담동": "1168010400", "삼성동": "1168010500",
        "대치동": "1168010600", "신사동": "1168010700",
        "논현동": "1168010800", "압구정동": "1168011000",
        "세곡동": "1168011100", "자곡동": "1168011200",
        "율현동": "1168011300", "일원동": "1168011400",
        "수서동": "1168011500", "도곡동": "1168011800",
    },
    "서초구": {
        "방배동": "1165010100", "양재동": "1165010200",
        "우면동": "1165010300", "원지동": "1165010400",
        "잠원동": "1165010600", "반포동": "1165010700",
        "서초동": "1165010800", "내곡동": "1165010900",
        "염곡동": "1165011000", "신원동": "1165011100",
    },
}


def build_session():
    """KB 공개 API 요청용 세션을 만든다."""
    import requests

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "ko-KR,ko;q=0.9",
        "Origin": "https://kbland.kr",
        "Referer": "https://kbland.kr/",
    })
    return session


def request_data(session, base_url: str, endpoint: str, params: dict[str, Any]) -> Any:
    """KB API를 재시도하고 응답의 실제 데이터만 반환한다."""
    last_error = None
    for attempt in range(6):
        try:
            response = session.get(f"{base_url}{endpoint}", params=params, timeout=30)
            response.raise_for_status()
            body = response.json().get("dataBody", {})
            if body.get("resultCode") == 33210:
                return []
            return body.get("data", [])
        except Exception as error:
            last_error = error
            time.sleep(0.8 * (attempt + 1))
    raise RuntimeError(f"KB API 요청 실패: {endpoint}, {params}") from last_error


def collect_complexes(session) -> pd.DataFrame:
    """강남·서초 법정동별 아파트 목록을 수집한다."""
    rows = []
    for district, dongs in LEGAL_DONGS.items():
        for dong, legal_code in dongs.items():
            items = request_data(
                session, API_COMPLEX, "/complexComm/hscmList",
                {"법정동코드": legal_code},
            )
            for item in items:
                if item.get("매물종별구분") != "01":
                    continue
                rows.append({
                    "자치구": district,
                    "동": dong,
                    "법정동코드": legal_code,
                    "단지기본일련번호": int(item["단지기본일련번호"]),
                    "아파트": item["단지명"],
                    "매물종별구분": item["매물종별구분"],
                    "매물종별구분명": item["매물종별구분명"],
                })
            time.sleep(0.05)
    complexes = (
        pd.DataFrame(rows)
        .drop_duplicates("단지기본일련번호")
        .sort_values(["자치구", "동", "아파트"])
        .reset_index(drop=True)
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    complexes.to_csv(COMPLEX_CACHE, index=False, encoding="utf-8-sig")
    return complexes


def add_reference_complex(complexes: pd.DataFrame) -> pd.DataFrame:
    """비교 기준인 송파구 리센츠를 조회 목록에 추가한다."""
    if REFERENCE_COMPLEX_ID in set(complexes["단지기본일련번호"]):
        return complexes
    reference = pd.DataFrame([{
        "자치구": "송파구", "동": "잠실동", "법정동코드": "1171010100",
        "단지기본일련번호": REFERENCE_COMPLEX_ID, "아파트": "리센츠",
        "매물종별구분": "01", "매물종별구분명": "아파트",
    }])
    return pd.concat([complexes, reference], ignore_index=True)


def get_monthly_price(session, complex_id: int, area_id: int) -> int | None:
    """면적 타입의 목표월 KB 매매 일반가(만원)를 반환한다."""
    data = request_data(
        session, API_PRICE, "/price/WholQuotList",
        {"단지기본일련번호": complex_id, "면적일련번호": area_id,
         "기준년": TARGET_YEAR_MONTH[:4]},
    )
    groups = data.get("시세", []) if isinstance(data, dict) else []
    for group in groups:
        for item in group.get("items", []):
            if item.get("기준년월") == TARGET_YEAR_MONTH:
                price = item.get("매매일반거래가")
                return int(price) if price else None
    return None


def collect_target_types(session, complexes: pd.DataFrame) -> pd.DataFrame:
    """전체 타입을 확인해 전용 83~85㎡와 목표월 일반가를 수집한다."""
    def collect_one(complex_row: pd.Series) -> list[dict[str, Any]]:
        worker_session = build_session()
        complex_id = int(complex_row["단지기본일련번호"])
        types = request_data(
            worker_session, API_COMPLEX, "/complexComm/typeList",
            {"단지기본일련번호": complex_id},
        )
        target_types = [
            item for item in types
            if AREA_MIN <= float(item.get("전용면적") or 0) <= AREA_MAX
        ]
        rows = []
        for item in target_types:
            area_id = int(item["면적일련번호"])
            rows.append({
                **complex_row.to_dict(),
                "면적일련번호": area_id,
                "공급면적_㎡": float(item["공급면적"]),
                "전용면적_㎡": float(item["전용면적"]),
                "주택형": item.get("주택형타입내용", ""),
                "KB일반가_만원": get_monthly_price(worker_session, complex_id, area_id),
                "시세기준월": TARGET_YEAR_MONTH,
            })
        return rows

    rows = []
    failures = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(collect_one, row): row for _, row in complexes.iterrows()}
        for completed, future in enumerate(as_completed(futures), 1):
            try:
                rows.extend(future.result())
            except RuntimeError:
                failures.append(futures[future])
            if completed % 100 == 0:
                print(f"단지 타입 조회: {completed:,}/{len(complexes):,}", flush=True)
    for row in failures:
        rows.extend(collect_one(row))
        time.sleep(0.2)
    type_prices = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    type_prices.to_csv(TYPE_PRICE_CACHE, index=False, encoding="utf-8-sig")
    return type_prices


def get_complex_detail(session, row: pd.Series) -> dict[str, Any]:
    """후보 단지의 세대수와 입주년월을 가져온다."""
    data = request_data(
        session, API_COMPLEX, "/complex/main",
        {"단지기본일련번호": int(row["단지기본일련번호"]),
         "매물종별구분": row["매물종별구분"],
         "면적일련번호": int(row["대표면적일련번호"])},
    )
    return data if isinstance(data, dict) else {}


def aggregate_complexes(session, type_prices: pd.DataFrame) -> pd.DataFrame:
    """복수 타입을 중위값으로 합치고 리센츠 대비 ±5% 후보를 만든다."""
    priced = type_prices.dropna(subset=["KB일반가_만원"]).copy()
    if priced.empty:
        raise ValueError("목표월 KB 일반가가 있는 전용 83~85㎡ 타입이 없습니다.")
    reference_prices = priced.loc[
        priced["단지기본일련번호"].eq(REFERENCE_COMPLEX_ID), "KB일반가_만원"
    ].tolist()
    if not reference_prices:
        raise ValueError("리센츠 전용 83~85㎡ 일반가가 없습니다.")
    reference_price = statistics.median(reference_prices)
    if reference_price != REFERENCE_PRICE_MANWON:
        raise ValueError(f"리센츠 기준가격이 예상과 다릅니다: {reference_price / 10_000:.2f}억")

    group_columns = [
        "자치구", "동", "단지기본일련번호", "아파트",
        "매물종별구분", "매물종별구분명",
    ]
    aggregated = priced.groupby(group_columns, as_index=False).agg(**{
        "KB일반가_만원": ("KB일반가_만원", "median"),
        "전용면적_㎡": ("전용면적_㎡", "median"),
        "대표면적일련번호": ("면적일련번호", "first"),
    })
    lower_price = reference_price * (1 - PRICE_TOLERANCE_RATE)
    upper_price = reference_price * (1 + PRICE_TOLERANCE_RATE)
    candidates = aggregated[
        aggregated["KB일반가_만원"].between(lower_price, upper_price)
        | aggregated["단지기본일련번호"].eq(REFERENCE_COMPLEX_ID)
    ].copy()

    details = []
    for _, row in candidates.iterrows():
        detail = get_complex_detail(session, row)
        details.append({
            "단지기본일련번호": int(row["단지기본일련번호"]),
            "세대수": detail.get("총세대수"),
            "입주년월": detail.get("입주년월"),
        })
        time.sleep(0.05)
    candidates = candidates.merge(pd.DataFrame(details), on="단지기본일련번호", how="left")
    candidates["아파트"] = candidates["아파트"].str.rstrip(".")
    candidates["준공연도"] = pd.to_numeric(
        candidates["입주년월"].astype(str).str[:4], errors="coerce"
    ).astype("Int64")
    candidates["KB시세_억원"] = candidates["KB일반가_만원"] / 10_000
    candidates["KB링크"] = candidates["단지기본일련번호"].map(
        lambda value: f"https://kbland.kr/se/c/{value}"
    )
    candidates["기준행"] = candidates["단지기본일련번호"].eq(REFERENCE_COMPLEX_ID)
    candidates["시세기준일"] = TARGET_PRICE_DATE
    result_columns = [
        "자치구", "동", "아파트", "준공연도", "세대수", "전용면적_㎡",
        "KB시세_억원", "시세기준일", "KB링크", "기준행",
    ]
    candidates = candidates[result_columns].reset_index(drop=True)
    candidates.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    return candidates


def run_full_scan() -> pd.DataFrame:
    """KB 자료를 새로 전수조사한다."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    session = build_session()
    complexes = collect_complexes(session)
    scan_complexes = add_reference_complex(complexes)
    print(f"조회 대상 전체 단지(리센츠 포함): {len(scan_complexes):,}개", flush=True)
    type_prices = collect_target_types(session, scan_complexes)
    return aggregate_complexes(session, type_prices)


def validate_data(apartments: pd.DataFrame) -> None:
    """기준 행, 중복, 면적, 기준일과 가격 범위를 검증한다."""
    reference_rows = apartments[apartments["기준행"]]
    if len(reference_rows) != 1:
        raise ValueError("리센츠 기준 행은 정확히 하나여야 합니다.")
    if apartments.duplicated(["자치구", "아파트"]).any():
        raise ValueError("단지별 중복 행이 있습니다.")
    if not apartments["전용면적_㎡"].between(83, 85).all():
        raise ValueError("전용면적 83~85㎡ 범위를 벗어난 단지가 있습니다.")
    if not apartments["시세기준일"].eq("2026-07-31").all():
        raise ValueError("KB시세 기준일이 2026년 7월 31일로 통일되지 않았습니다.")
    comparison = apartments[~apartments["기준행"]]
    outside = comparison[
        ~comparison["KB시세_억원"].between(LOWER_PRICE_EOK, UPPER_PRICE_EOK)
    ]
    if not outside.empty:
        raise ValueError("±5% 범위를 벗어난 단지가 있습니다: " + ", ".join(outside["아파트"]))


def format_korean_price(price_eok: float) -> str:
    """억원 단위 값을 억·만원 형식으로 표시한다."""
    total_manwon = round(price_eok * 10_000)
    eok, manwon = divmod(total_manwon, 10_000)
    return f"{eok}억 {manwon:,}만원" if manwon else f"{eok}억원"


def build_table_html(apartments: pd.DataFrame) -> str:
    """KB 링크와 기준 행을 포함한 비교 HTML 표를 만든다."""
    rows = []
    for _, row in apartments.iterrows():
        difference_rate = row["KB시세_억원"] / REFERENCE_PRICE_EOK - 1
        difference_text = "기준" if row["기준행"] else f"{difference_rate:+.2%}"
        row_class = ' class="reference-row"' if row["기준행"] else ""
        rows.append(
            f"<tr{row_class}>"
            f"<td class='identifier'>{row['자치구']}</td>"
            f"<td class='identifier'>{row['동']}</td>"
            f"<td class='row-label'><a href='{row['KB링크']}' target='_blank' rel='noopener'>{row['아파트']}</a></td>"
            f"<td class='identifier'>{row['준공연도']}</td>"
            f"<td class='number'>{row['세대수']:,}</td>"
            f"<td class='number'>{row['전용면적_㎡']:.2f}</td>"
            f"<td class='number'>{format_korean_price(row['KB시세_억원'])}</td>"
            f"<td class='number'>{difference_text}</td>"
            "</tr>"
        )
    return f"""
<style>
.recenz-table-section {{ max-width:700px; margin:12px 0 28px; font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; }}
.recenz-table-brand {{ margin:0 0 5px; color:#64748b; font-size:13px; font-weight:400; }}
.recenz-table-title {{ margin:0 0 4px; color:#0f172a; font-size:20px; line-height:1.3; font-weight:700; }}
.recenz-table-caption {{ margin:0 0 14px; color:#64748b; font-size:13px; line-height:1.45; }}
.recenz-table-wrap {{ overflow:hidden; border:1px solid #f0f2f5; border-radius:12px; background:#fff; }}
.recenz-result-table {{ width:100%; table-layout:fixed; border-collapse:separate; border-spacing:0; color:#1e293b; font-size:13px; font-variant-numeric:tabular-nums; }}
.recenz-result-table th {{ padding:10px 2px; background:#2b4a75; color:white; font-weight:700; line-height:1.3; text-align:center; word-break:keep-all; }}
.recenz-result-table td {{ padding:9px 3px; border-bottom:1px solid #f0f2f5; background:white; line-height:1.4; white-space:nowrap; }}
.recenz-result-table tbody tr:nth-child(even) td {{ background:#fafbfc; }}
.recenz-result-table tbody tr:hover td {{ background:#eff6ff; }}
.recenz-result-table tbody tr:last-child td {{ border-bottom:0; }}
.recenz-result-table .identifier {{ text-align:center; }}
.recenz-result-table .number {{ text-align:right; }}
.recenz-result-table .row-label {{ overflow:hidden; text-align:left; text-overflow:ellipsis; font-weight:600; }}
.recenz-result-table a {{ color:#1e293b; text-decoration:none; border-bottom:1px solid #94a3b8; }}
.recenz-result-table .reference-row td {{ background:#eaf2fc !important; border-top:2px solid #2f7dd3; border-bottom:2px solid #c5d9f4; font-weight:700; }}
.recenz-result-table th:nth-child(1) {{ width:8%; }}
.recenz-result-table th:nth-child(2) {{ width:9%; }}
.recenz-result-table th:nth-child(3) {{ width:23%; }}
.recenz-result-table th:nth-child(4) {{ width:9%; }}
.recenz-result-table th:nth-child(5) {{ width:11%; }}
.recenz-result-table th:nth-child(6) {{ width:10%; }}
.recenz-result-table th:nth-child(7) {{ width:17%; }}
.recenz-result-table th:nth-child(8) {{ width:13%; }}
.recenz-table-footnote {{ margin:9px 0 0; color:#64748b; font-size:13px; line-height:1.5; }}
</style>
<section class='recenz-table-section'>
  <p class='recenz-table-brand'>대도시 연구실</p>
  <h2 class='recenz-table-title'>리센츠 가격으로 살 수 있는 강남·서초 아파트</h2>
  <p class='recenz-table-caption'>전용 83~85㎡ · 2026년 7월 KB 일반가 · 리센츠 중위값 33.5억 대비 ±5%</p>
  <div class='recenz-table-wrap'>
    <table class='recenz-result-table'>
      <thead><tr><th>지역</th><th>동</th><th>아파트</th><th>준공</th><th>세대수</th><th>전용㎡</th><th>KB 일반가</th><th>리센츠 대비</th></tr></thead>
      <tbody>{''.join(rows)}</tbody>
    </table>
  </div>
  <p class='recenz-table-footnote'>※ 모든 단지의 복수 타입은 일반가 중위값으로 통합하며, 리센츠도 같은 기준을 적용했습니다.<br>※ 단지명을 누르면 KB부동산 시세 페이지가 열립니다.</p>
</section>
"""


def build_dong_choices_table_html() -> str:
    """리센츠 가격으로 선택할 수 있는 강남·서초 동별 단지를 표시한다."""
    choice_rows = [
        ("강남구", "개포동", "디퍼아 · 개포래미안포레스트 · 래미안블레스티지", "대표 신축 대단지까지 가능"),
        ("강남구", "대치동", "대치삼성(래미안)", "진입 가능하지만 선택 제한"),
        ("강남구", "도곡동", "개포한신 · 도곡동삼성래미안", "주요 아파트 선택 가능"),
        ("강남구", "삼성동", "삼성동힐스테이트1·2단지 · 롯데캐슬프레미어", "선호도 높은 주요 단지 가능"),
        ("강남구", "역삼동", "개나리SK뷰 · 개나리푸르지오 · 테헤란아이파크", "주요 아파트 선택 가능"),
        ("강남구", "일원동", "디에이치포레센트 · 우성7차 · 가람", "대표 신축·재건축 추진 단지까지 가능"),
        ("강남구", "청담동", "진흥", "진입 가능하지만 선택 제한"),
        ("서초구", "방배동", "방배그랑자이", "대표 신축 대단지 가능"),
        ("서초구", "서초동", "에스티지S · 푸르지오써밋 · 에스티지", "주요 준신축 단지 가능"),
        ("서초구", "잠원동", "잠원한신 · 한강 · 신반포16차", "진입 가능하지만 소규모 재건축 단지 중심"),
    ]
    rows = "".join(
        f"<tr><td>{district}</td><td>{dong}</td><td>{choices}</td><td>{feature}</td></tr>"
        for district, dong, choices, feature in choice_rows
    )
    return f"""
<style>
.recenz-choice-table tbody td:nth-child(1),
.recenz-choice-table tbody td:nth-child(2) {{ text-align:center; }}
.recenz-choice-table tbody td:nth-child(3),
.recenz-choice-table tbody td:nth-child(4) {{ text-align:left; }}
</style>
<section class='recenz-table-section' style='margin-top:28px'>
  <p class='recenz-table-brand'>대도시 연구실</p>
  <h3 class='recenz-table-title' style='font-size:20px'>리센츠 가격으로 살 수 있는 강남·서초 동별 선택지</h3>
  <div class='recenz-table-wrap'>
    <table class='recenz-result-table recenz-choice-table'>
      <colgroup><col style='width:12%'><col style='width:12%'><col style='width:43%'><col style='width:33%'></colgroup>
      <thead><tr><th style='width:12%'>지역</th><th style='width:12%'>동</th><th style='width:43%'>대표 선택지</th><th style='width:33%'>선택지 특징</th></tr></thead>
      <tbody>{rows}</tbody>
    </table>
  </div>
</section>
"""


def save_csv(apartments: pd.DataFrame) -> Path:
    """계산용 숫자와 KB 링크를 CSV로 저장한다."""
    output = apartments.copy()
    output["리센츠_대비"] = output["KB시세_억원"] / REFERENCE_PRICE_EOK - 1
    output.loc[output["기준행"], "리센츠_대비"] = 0.0
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "recenz_vs_gangnam_kb_price.csv"
    output.to_csv(output_path, index=False, encoding="utf-8-sig")
    return output_path


def build_survey_summary_html(
    apartments: pd.DataFrame,
    complexes: pd.DataFrame,
    type_prices: pd.DataFrame,
) -> str:
    """저장된 CSV에서 전수조사 규모와 선별 결과를 계산한다."""
    reference = apartments[apartments["기준행"]].iloc[0]
    reference_price = float(reference["KB시세_억원"])
    reference_type_count = int(
        type_prices["단지기본일련번호"].eq(REFERENCE_COMPLEX_ID).sum()
    )
    non_reference_types = type_prices[
        type_prices["단지기본일련번호"].ne(REFERENCE_COMPLEX_ID)
    ]
    complex_count = complexes["단지기본일련번호"].nunique()
    target_complex_count = non_reference_types["단지기본일련번호"].nunique()
    priced_complex_count = non_reference_types.dropna(
        subset=["KB일반가_만원"]
    )["단지기본일련번호"].nunique()
    candidate_count = int((~apartments["기준행"]).sum())
    lower_price = reference_price * (1 - PRICE_TOLERANCE_RATE)
    upper_price = reference_price * (1 + PRICE_TOLERANCE_RATE)
    return f"""
<section class='recenz-table-section' style='margin-bottom:14px'>
  <h2 class='recenz-table-title'>조사 결과</h2>
  <ul style='margin:8px 0 0; padding-left:24px; color:#525252; font-size:14px; line-height:1.55'>
    <li>강남·서초 아파트: {complex_count:,}개 단지</li>
    <li>전용 83~85㎡ 타입 발견: {target_complex_count:,}개 단지</li>
    <li>해당 면적의 2026년 7월 KB 일반가 보유: {priced_complex_count:,}개 단지</li>
    <li>리센츠 기준가격: 전용 83~85㎡ {reference_type_count}개 타입의 중위값 {reference_price:g}억 원</li>
    <li>검색 범위: {lower_price:g}억~{upper_price:g}억 원</li>
    <li>최종 후보: {candidate_count:,}개 단지 + 리센츠 기준행</li>
  </ul>
</section>
"""


def main() -> None:
    """저장 결과를 표시하거나 명시적으로 새 전수조사를 실행한다."""
    refresh_requested = REFRESH_KB_DATA is True
    if refresh_requested:
        print("실행 모드: 새로 전수조사 ON — KB 전체 단지 조회를 시작합니다.")
        apartments = run_full_scan()
    elif (
        RESULT_PATH.exists()
        and COMPLEX_CACHE.exists()
        and TYPE_PRICE_CACHE.exists()
    ):
        print("실행 모드: 새로 전수조사 OFF — 저장된 CSV만 읽습니다.")
        apartments = pd.read_csv(RESULT_PATH)
    else:
        print("실행 모드: 새로 전수조사 OFF — 네트워크를 호출하지 않습니다.")
        display(HTML("""
<div style='max-width:700px; padding:16px 18px; border:1px solid #f3c7c7;
            border-radius:12px; background:#fff7f7; color:#991b1b;
            font-family:Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;'>
  <strong>저장된 전수조사 결과가 없습니다.</strong><br>
  <code>새로 전수조사</code> 옵션을 ON으로 바꾼 뒤 다시 실행해 주세요.
</div>
"""))
        return
    complexes = pd.read_csv(COMPLEX_CACHE, dtype={"법정동코드": str})
    type_prices = pd.read_csv(TYPE_PRICE_CACHE, dtype={"법정동코드": str})
    apartments = apartments.sort_values(
        ["기준행", "자치구", "동", "KB시세_억원", "아파트"],
        ascending=[False, True, True, False, True],
    ).reset_index(drop=True)
    validate_data(apartments)
    csv_path = save_csv(apartments)
    summary_html = build_survey_summary_html(apartments, complexes, type_prices)
    display(HTML(
        summary_html
        + build_table_html(apartments)
        + build_dong_choices_table_html()
    ))
    display(FileLink(str(csv_path), result_html_prefix="CSV 다운로드: "))


main()
